# VitiGuard MD — фильтр «лист / не лист» на полевых снимках (ml/fine-01)

Прежний фильтр отсеивал снимки куста с несколькими листьями как «не виноград». Блокнот заново обучает только фильтр (модель болезней не меняется):

- скачивает открытые наборы под CC BY 4.0: PlantVillage, NGLD, HERMOS, Tardif et al. (Flavescence dorée и эска в поле), Imagenette;
- берёт ряды INESC из `Мой диск/vitiguard/vine/vine_dataset.zip`;
- сравнивает новый фильтр со старым на отложенных снимках и пишет результат в `Мой диск/vitiguard/gate`.

**Перед запуском:** положите в `Мой диск/vitiguard/models` три файла с сервера: `diagnosis.onnx`, `diagnosis.json`, `diagnosis.gate.json`. GPU не нужен. «Среда выполнения» → «Выполнить все». Около 40 минут, большая часть — скачивание (около 7 ГБ).

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/vitiguard"
MODELS = f"{DRIVE}/models"
OUT = f"{DRIVE}/gate"
BRANCH = "main"

need = ["diagnosis.onnx", "diagnosis.json", "diagnosis.gate.json"]
missing = [name for name in need if not os.path.exists(f"{MODELS}/{name}")]
assert not missing, f"Нет файлов в Мой диск/vitiguard/models: {missing}"
assert os.path.exists(f"{DRIVE}/vine/vine_dataset.zip"), "Нет vine/vine_dataset.zip"
print("Файлы модели на месте")

In [ ]:
!rm -rf /content/VitiGuard-MD_back
!git clone -q --depth 1 --branch {BRANCH} https://github.com/androtschii/VitiGuard-MD_back /content/VitiGuard-MD_back
%cd /content/VitiGuard-MD_back/ml
!pip install -q -e . onnx onnxruntime
!mkdir -p /content/model && cp "{MODELS}"/diagnosis.* /content/model/

## Данные

Архивы распаковываются на диск Colab. Из больших полевых снимков сразу делаются копии до 1600 точек по длинной стороне: так хватает места и быстрее считать.

In [ ]:
import io
import tarfile
import urllib.request
import zipfile
from pathlib import Path

from PIL import Image

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)
SUFFIXES = (".jpg", ".jpeg", ".png", ".webp")

def fetch(url, target):
    target = Path(target)
    if not target.exists():
        print("скачивание", target.name, flush=True)
        urllib.request.urlretrieve(url, target)
    return target

def frames_from_zip(source, out, limit=1500, max_side=1600):
    """Снимки из архива, в том числе из вложенных архивов, уменьшенные копии."""
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    count = 0
    def walk(zip_file):
        nonlocal count
        for name in zip_file.namelist():
            if count >= limit:
                return
            lower = name.lower()
            if lower.endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(zip_file.read(name))) as inner:
                    walk(inner)
            elif lower.endswith(SUFFIXES) and "__macosx" not in lower:
                try:
                    image = Image.open(io.BytesIO(zip_file.read(name))).convert("RGB")
                except Exception:
                    continue
                image.thumbnail((max_side, max_side))
                image.save(out / f"{count:05d}.jpg", quality=85)
                count += 1
    with zipfile.ZipFile(source) as zip_file:
        walk(zip_file)
    print(out.name, count, "снимков")
    return out

MENDELEY = "https://data.mendeley.com/public-api/zip/{}/download/{}"

In [ ]:
# Листья по одному и листья других культур — PlantVillage, только нужные папки
GRAPE = ["Grape___Black_rot", "Grape___Esca_(Black_Measles)",
         "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy"]
OTHER = ["Apple___healthy", "Apple___Apple_scab", "Tomato___healthy",
         "Tomato___Early_blight", "Corn_(maize)___healthy", "Potato___healthy",
         "Soybean___healthy", "Strawberry___healthy", "Cherry_(including_sour)___healthy"]
PV = DATA / "plantvillage"
if not PV.exists():
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git {PV}
folders = " ".join(f'"raw/color/{name}"' for name in GRAPE + OTHER)
!git -C {PV} sparse-checkout set {folders}

In [ ]:
from vitiguard_ml.data import hermos, ngld

ngld_root = ngld.download(DATA / "ngld")
hermos_zip = hermos.download(DATA / "hermos")
rows, skipped = hermos.prepare(hermos_zip, DATA / "hermos_crops")
print("вырезки HERMOS:", len(rows))
frames_from_zip(hermos_zip, DATA / "canopy_hermos", limit=600)
tardif_zip = fetch(MENDELEY.format("3dr9r3w3jn", 2), DATA / "tardif.zip")
frames_from_zip(tardif_zip, DATA / "canopy_tardif", limit=1500)
frames_from_zip(f"{DRIVE}/vine/vine_dataset.zip", DATA / "canopy_inesc", limit=1500)
imagenette = fetch("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz", DATA / "imagenette.tgz")
if not (DATA / "imagenette2-160").exists():
    with tarfile.open(imagenette) as archive:
        archive.extractall(DATA)

## Обучение фильтра и сравнение со старым

In [ ]:
lab = " ".join(f'"{PV}/raw/color/{name}"' for name in GRAPE) + f' "{ngld_root}"'
other = " ".join(f'"crops={PV}/raw/color/{name}"' for name in OTHER)
!python -m vitiguard_ml.gate_field --model /content/model/diagnosis.onnx     --lab {lab} --crops {DATA}/hermos_crops     --canopy hermos={DATA}/canopy_hermos tardif={DATA}/canopy_tardif inesc={DATA}/canopy_inesc     --not-grape imagenette={DATA}/imagenette2-160 {other}     --out /content/gate

In [ ]:
import json
import shutil

os.makedirs(OUT, exist_ok=True)
for name in ["diagnosis.gate.json", "gate_report.json"]:
    shutil.copy(f"/content/gate/{name}", f"{OUT}/{name}")
report = json.loads(Path("/content/gate/gate_report.json").read_text())
keys = sorted({key for row in report.values() for key in row})
print(f"{'':<22}" + "".join(f"{name:>8}" for name in report))
for key in keys:
    print(f"{key:<22}" + "".join(f"{report[name].get(key, float('nan')):>8.3f}" for name in report))
print("Готово: скачайте папку Мой диск/vitiguard/gate")